In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import category_encoders as ce

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.preprocessing import LabelEncoder,StandardScaler,Normalizer
from sklearn.metrics import f1_score,r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer

import seaborn as sns
import matplotlib.pyplot as plt


import warnings
warnings.filterwarnings('ignore')

In [13]:
data = pd.read_csv('../data/favorita-grocery-sales-forecasting/train.csv')
# Path сам склеит части пути правильным для текущей ОС образом
#file_path = Path('data') / 'favorita-grocery-sales-forecasting' / 'train.csv'
#data = pd.read_csv(file_path)
data.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [14]:
data.describe()

,id,store_nbr,item_nbr,unit_sales
count,1.254970e+08,1.254970e+08,1.254970e+08,1.254970e+08
mean,6.274852e+07,2.746458e+01,9.727692e+05,8.554865e+00
std,3.622788e+07,1.633051e+01,5.205336e+05,2.360515e+01
min,0.000000e+00,1.000000e+00,9.699500e+04,-1.537200e+04
25%,3.137426e+07,1.200000e+01,5.223830e+05,2.000000e+00
50%,6.274852e+07,2.800000e+01,9.595000e+05,4.000000e+00
75%,9.412278e+07,4.300000e+01,1.354380e+06,9.000000e+00
max,1.254970e+08,5.400000e+01,2.127114e+06,8.944000e+04


In [15]:
data.dtypes

id               int64
date               str
store_nbr        int64
item_nbr         int64
unit_sales     float64
onpromotion     object
dtype: object

In [17]:
nan_count = data['onpromotion'].isna().sum()
print(nan_count)

21657651


In [19]:
data.shape[0]

125497040

In [20]:
125497040 - 21657651

103839389

In [ ]:
# Выдаст одну случайную строку, где нет пропуска
primer = data[data['onpromotion'].notna()].sample(1)
print(primer)

                  id        date  store_nbr  item_nbr  unit_sales onpromotion
113844590  113844590  2017-04-28          2    655427        14.0        True


In [23]:
data[data['onpromotion'].notna()].shape

(103839389, 6)

In [24]:
print(data['onpromotion'].value_counts())

onpromotion
False    96028767
True      7810622
Name: count, dtype: int64


In [25]:
data['onpromotion'] = data['onpromotion'].map({True: 1, False: 0}).fillna(-1).astype(int)

In [26]:
print(data['onpromotion'].value_counts())

onpromotion
 0    96028767
-1    21657651
 1     7810622
Name: count, dtype: int64


In [31]:
data['date'] = pd.to_datetime(data['date'])

In [32]:
data.dtypes

id                      int64
date           datetime64[us]
store_nbr               int64
item_nbr                int64
unit_sales            float64
onpromotion             int64
dtype: object

In [33]:
data.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,-1
1,1,2013-01-01,25,105574,1.0,-1
2,2,2013-01-01,25,105575,2.0,-1
3,3,2013-01-01,25,108079,1.0,-1
4,4,2013-01-01,25,108701,1.0,-1


In [34]:
#check for repeated data points using article_id
data = data.drop_duplicates(subset=['id'])